# 01a - Reuters Anotasyon Seti Hazırlığı


In [1]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ============================================================
# aggregated_financial_news_enriched.csv OKU
# ============================================================

DB_DIR = DATA_ROOT

# Dosyayı otomatik bul
NEWS_PATH = paths.AGGREGATED_FINANCIAL_NEWS_PATH

if not NEWS_PATH.exists():
    print("DB klasöründeki CSV dosyaları:")
    for p in DB_DIR.rglob("*.csv"):
        print("-", p)
    raise FileNotFoundError("aggregated_financial_news_enriched.csv bulunamadı.")


print("Okunan dosya:")
print(NEWS_PATH)

# Excel CSV bazen encoding sorunlu olabilir; birkaç encoding deneyelim
encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]

news_final_df = None
used_encoding = None

for enc in encodings:
    try:
        news_final_df = pd.read_csv(NEWS_PATH, encoding=enc)
        used_encoding = enc
        break
    except UnicodeDecodeError:
        pass

if news_final_df is None:
    raise ValueError("CSV dosyası hiçbir encoding ile okunamadı.")

print("\nKullanılan encoding:", used_encoding)
print("news_final_df shape:", news_final_df.shape)

print("\nColumns:")
print(news_final_df.columns.tolist())



Okunan dosya:
D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\processed\aggregated_financial_news_enriched.csv

Kullanılan encoding: utf-8
news_final_df shape: (943326, 12)

Columns:
['date', 'text', 'sentiment', 'dataset', 'hour', 'date_only', 'primary_topic', 'topics', 'topic_confidence', 'financial_score', 'sentiment_confidence', 'sentiment_score']


In [2]:
# ============================================================
# news_final_df içinde 5000'den fazla örneği olan datasetleri al
# ============================================================

if "news_final_df" not in globals():
    raise ValueError("news_final_df bulunamadı. Önce aggregated_financial_news_enriched.csv okuma hücresini çalıştır.")

if "dataset" not in news_final_df.columns:
    raise ValueError("news_final_df içinde 'dataset' kolonu yok.")

# Dataset sayıları
dataset_counts = news_final_df["dataset"].value_counts(dropna=False)

# 5000'den fazla örneği olan dataset isimleri
valid_datasets = dataset_counts[dataset_counts > 5000].index.tolist()

print("5000'den fazla örneği olan datasetler:")
print(dataset_counts[dataset_counts > 5000])

# Sadece bu datasetlere ait satırları al
news_over_5000_df = news_final_df[
    news_final_df["dataset"].isin(valid_datasets)
].copy().reset_index(drop=True)

print("\nnews_over_5000_df shape:", news_over_5000_df.shape)

print("\nFiltre sonrası dataset dağılımı:")
print(news_over_5000_df["dataset"].value_counts())

5000'den fazla örneği olan datasetler:
dataset
daily_stock_news    715455
crypto_news         201365
bitcoin_tweets       20073
reuters_news          6433
Name: count, dtype: int64

news_over_5000_df shape: (943326, 12)

Filtre sonrası dataset dağılımı:
dataset
daily_stock_news    715455
crypto_news         201365
bitcoin_tweets       20073
reuters_news          6433
Name: count, dtype: int64


In [3]:
news_over_5000_df

,date,text,sentiment,dataset,hour,date_only,primary_topic,topics,topic_confidence,financial_score,sentiment_confidence,sentiment_score
0,2006-10-20 00:00:00,-- exxon mobil offers plan to end alaska dispu...,positive,reuters_news,0,2006-10-20,war,['war'],0.380579,0.441421,0.555539,1.0
1,2006-10-20 00:00:00,"-- hey buddy, can you spare $600 for a google ...",positive,reuters_news,0,2006-10-20,stock,['stock' 'finance'],0.400924,0.390339,0.919182,1.0
2,2006-10-21 00:00:00,-- aol ceo says sales may shrink for two years...,negative,reuters_news,0,2006-10-21,stock,['stock' 'finance'],0.268268,0.244141,0.967526,-1.0
3,2006-10-22 00:00:00,"-- fed to keep hawkish tone, hold rates steady...",neutral,reuters_news,0,2006-10-22,macro,['macro'],0.530542,0.535168,0.807155,0.0
4,2006-10-22 00:00:00,-- pluspetrol says losing $2.4 mln/day in peru...,negative,reuters_news,0,2006-10-22,war,['war'],0.277602,0.289422,0.967845,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
943321,2025-12-03 10:15:22,kumining elevates cloud mining ecosystem with ...,positive,crypto_news,10,2025-12-03,crypto,['crypto' 'finance'],0.322861,0.226318,0.822513,1.0
943322,2025-12-03 10:15:49,major us bank recommends wealthy clients consi...,neutral,crypto_news,10,2025-12-03,crypto,['crypto' 'finance'],0.308155,0.255369,0.814540,0.0
943323,2025-12-03 10:17:34,heres bitcoins next big target after $93k brea...,neutral,crypto_news,10,2025-12-03,finance,[],0.191629,0.287040,0.870270,0.0
943324,2025-12-03 10:23:01,bitcoin short-term holder shakeout could accel...,positive,crypto_news,10,2025-12-03,crypto,['crypto' 'finance'],0.266236,0.292564,0.941623,1.0


In [4]:
# ============================================================
# REUTERS NEWS - 5000 DENGELİ RANDOM HEADLINE SAMPLE
#
# Amaç:
# - news_final_df içinden sadece dataset == reuters_news seçilir.
# - Reuters tam haber metninden sadece ilk başlık satırı çıkarılır.
# - Confidence dikkate alınmaz.
# - Sadece sentiment_score: -1 / 0 / 1 sınıflarına göre mümkün olduğunca dengeli seçilir.
# - Annotation master + 50'şer batch CSV/TXT oluşturulur.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import math

# ------------------------------------------------------------
# 0) Kontrol ve ayarlar
# ------------------------------------------------------------
if "news_final_df" not in globals():
    raise ValueError("news_final_df bulunamadı. Önce aggregated_financial_news_enriched.csv okuma hücresini çalıştır.")

PROJECT_DIR = PROJECT_ROOT
DB_DIR = DATA_ROOT

OUT_DIR = paths.REUTERS_ANNOTATION_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_DIR = paths.REUTERS_ANNOTATION_BATCHES_DIR
BATCH_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLE = 5000
BATCH_SIZE = 50

# Eski hatalı batch dosyalarını temizlemek istersen True kalsın.
CLEAR_OLD_OUTPUTS = True

if CLEAR_OLD_OUTPUTS:
    for p in BATCH_DIR.glob("reuters_annotation_batch_*.*"):
        p.unlink()

    for p in [
        OUT_DIR / "reuters_annotation_5000_master.csv",
        OUT_DIR / "reuters_annotation_5000_master.xlsx",
        OUT_DIR / "reuters_annotation_5000_master.parquet",
    ]:
        if p.exists():
            p.unlink()

print("OUT_DIR:", OUT_DIR)
print("BATCH_DIR:", BATCH_DIR)

# ------------------------------------------------------------
# 1) Reuters headline çıkarma fonksiyonu
# ------------------------------------------------------------
def extract_reuters_headline(text):
    """
    Reuters tam haber metninden ilk anlamlı başlık satırını çıkarır.

    Örnek ham metin:
    -- microsoft to start vista coupon plan for pc buyers
    --
    -- tue oct 24, 2006 704pm edt
    --
    seattle reuters - ...

    Çıktı:
    microsoft to start vista coupon plan for pc buyers
    """

    if pd.isna(text):
        return ""

    text = str(text).strip()

    if not text or text.lower() == "nan":
        return ""

    lines = text.splitlines()
    clean_lines = []

    for line in lines:
        line = str(line).strip()

        if not line:
            continue

        # Başındaki Reuters tirelerini temizle
        line = re.sub(r"^\s*[-–—]+\s*", "", line).strip()

        if not line:
            continue

        low = line.lower()

        # Sadece tarih/saat olan satırları at
        # Örn: tue oct 24, 2006 704pm edt
        has_weekday = bool(re.search(r"\b(mon|tue|wed|thu|fri|sat|sun)\b", low))
        has_month = bool(re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", low))
        has_year_or_time = bool(re.search(r"\b\d{4}\b|\b\d{1,2}:\d{2}\b|\b\d{3,4}\s*(am|pm)\b|\b(am|pm)\b", low))

        if has_weekday and has_month and has_year_or_time:
            continue

        # Reuters gövde başlangıcını başlık olarak alma
        if "reuters -" in low:
            continue

        # Çok teknik/boş meta satırları at
        if low in ["reuters", "update", "brief", "corrected"]:
            continue

        clean_lines.append(line)

    if not clean_lines:
        return ""

    headline = clean_lines[0]

    # Fazla boşlukları temizle
    headline = re.sub(r"\s+", " ", headline).strip()

    # Başta/sonda kalan gereksiz tireleri temizle
    headline = re.sub(r"^\s*[-–—]+\s*", "", headline).strip()
    headline = re.sub(r"\s*[-–—]+\s*$", "", headline).strip()

    return headline


# ------------------------------------------------------------
# 2) Sadece Reuters seç
# ------------------------------------------------------------
reuters_df = news_final_df[
    news_final_df["dataset"].astype(str).str.lower().str.strip() == "reuters_news"
].copy()

print("Raw reuters_df shape:", reuters_df.shape)

if len(reuters_df) == 0:
    raise ValueError("dataset == reuters_news olan satır bulunamadı.")

# Orijinal indexi sakla
reuters_df["source_row_index"] = reuters_df.index

# ------------------------------------------------------------
# 3) Tam metinden headline çıkar
# ------------------------------------------------------------
if "text" not in reuters_df.columns:
    raise ValueError("news_final_df içinde text kolonu yok.")

reuters_df["raw_text"] = reuters_df["text"].astype(str).str.strip()
reuters_df["headline_text"] = reuters_df["raw_text"].apply(extract_reuters_headline)

reuters_df["headline_text"] = (
    reuters_df["headline_text"]
    .astype(str)
    .str.strip()
)

# Headline uzunlukları
reuters_df["n_words"] = reuters_df["headline_text"].str.split().str.len()
reuters_df["text_len"] = reuters_df["headline_text"].str.len()

# ------------------------------------------------------------
# 4) Headline temizlik
# ------------------------------------------------------------
reuters_headline_df = reuters_df[
    reuters_df["headline_text"].notna()
    & (reuters_df["headline_text"] != "")
    & (reuters_df["headline_text"].str.lower() != "nan")
    & (reuters_df["n_words"] >= 4)
    & (reuters_df["text_len"] >= 20)
    & (reuters_df["text_len"] <= 180)
].copy()

# Bozuk encoding / anlamsız kalıntıları çıkar
bad_patterns = [
    r"â",                  # encoding bozukluğu
    r"^\s*[-–—]+\s*$",     # sadece tire
    r"^reuters$",          # sadece reuters
]

bad_regex = "|".join(bad_patterns)

reuters_headline_df = reuters_headline_df[
    ~reuters_headline_df["headline_text"]
    .str.lower()
    .str.contains(bad_regex, regex=True, na=False)
].copy()

# Duplicate temizliği
reuters_headline_df["headline_norm"] = (
    reuters_headline_df["headline_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

reuters_headline_df = reuters_headline_df.drop_duplicates(
    subset=["headline_norm"]
).reset_index(drop=True)

print("Headline Reuters shape:", reuters_headline_df.shape)

display(
    reuters_headline_df[
        ["date", "headline_text", "sentiment", "sentiment_score", "sentiment_confidence", "n_words", "text_len"]
    ].head(PREVIEW_ROWS)
)

# ------------------------------------------------------------
# 5) Sentiment score normalize
# ------------------------------------------------------------
if "sentiment_score" not in reuters_headline_df.columns:
    raise ValueError("sentiment_score kolonu yok. -1 / 0 / 1 dengesine göre seçim yapılamaz.")

reuters_headline_df["sentiment_score"] = pd.to_numeric(
    reuters_headline_df["sentiment_score"],
    errors="coerce"
)

reuters_headline_df = reuters_headline_df[
    reuters_headline_df["sentiment_score"].isin([-1, 0, 1])
].copy()

score_to_label = {
    -1: "negative",
     0: "neutral",
     1: "positive",
}

reuters_headline_df["finbert_score"] = reuters_headline_df["sentiment_score"].astype(int)
reuters_headline_df["finbert_label"] = reuters_headline_df["finbert_score"].map(score_to_label)

if "sentiment_confidence" in reuters_headline_df.columns:
    reuters_headline_df["finbert_confidence"] = pd.to_numeric(
        reuters_headline_df["sentiment_confidence"],
        errors="coerce"
    )
else:
    reuters_headline_df["finbert_confidence"] = np.nan

print("\nReuters headline score distribution:")
print(reuters_headline_df["finbert_score"].value_counts().sort_index())

print("\nReuters headline label distribution:")
print(reuters_headline_df["finbert_label"].value_counts())

if len(reuters_headline_df) < N_SAMPLE:
    raise ValueError(
        f"Headline temizlikten sonra {N_SAMPLE} örnek kalmadı. "
        f"Mevcut: {len(reuters_headline_df)}"
    )

# ------------------------------------------------------------
# 6) 5000 örneği -1 / 0 / 1'e göre mümkün olduğunca dengeli random seç
# Confidence dikkate alınmaz.
# ------------------------------------------------------------
available_counts = reuters_headline_df["finbert_score"].value_counts().sort_index()

print("\nAvailable score counts:")
print(available_counts)

neutral_available = int((reuters_headline_df["finbert_score"] == 0).sum())
negative_available = int((reuters_headline_df["finbert_score"] == -1).sum())
positive_available = int((reuters_headline_df["finbert_score"] == 1).sum())

# İdeal hedefler
ideal_each = N_SAMPLE // 3

# Neutral azsa tamamını değil, ideal kadar varsa ideal kadar al.
# Eğer neutral idealden azsa tüm neutral alınır.
neutral_n = min(neutral_available, ideal_each)

remaining = N_SAMPLE - neutral_n

negative_n = remaining // 2
positive_n = remaining - negative_n

if negative_available < negative_n:
    raise ValueError(
        f"negative için yeterli örnek yok. "
        f"Gerekli: {negative_n}, mevcut: {negative_available}"
    )

if positive_available < positive_n:
    raise ValueError(
        f"positive için yeterli örnek yok. "
        f"Gerekli: {positive_n}, mevcut: {positive_available}"
    )

target_counts = {
    -1: negative_n,
     0: neutral_n,
     1: positive_n,
}

print("\nTarget counts:")
print(target_counts)

sample_parts = []

for score, target_n in target_counts.items():
    temp = reuters_headline_df[reuters_headline_df["finbert_score"] == score].copy()

    sampled = temp.sample(
        n=target_n,
        random_state=RANDOM_STATE
    )

    sample_parts.append(sampled)

sample_df = pd.concat(sample_parts, ignore_index=True)

# Son sırayı karıştır
sample_df = sample_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("\nSelected sample shape:", sample_df.shape)

print("\nSelected score distribution:")
print(sample_df["finbert_score"].value_counts().sort_index())

print("\nSelected label distribution:")
print(sample_df["finbert_label"].value_counts())

# ------------------------------------------------------------
# 7) Annotation master oluştur
# ------------------------------------------------------------
annotation_df = pd.DataFrame()

annotation_df["annotation_id"] = [
    f"REUTERS_ANN_{i+1:05d}" for i in range(len(sample_df))
]

annotation_df["source_dataset"] = "reuters_news"
annotation_df["source_row_index"] = sample_df["source_row_index"].values

if "date" in sample_df.columns:
    annotation_df["date"] = pd.to_datetime(sample_df["date"], errors="coerce")
else:
    annotation_df["date"] = pd.NaT

if "date_only" in sample_df.columns:
    annotation_df["date_only"] = sample_df["date_only"].values
else:
    annotation_df["date_only"] = pd.NaT

# Asıl önemli düzeltme burada:
# text_en artık full text değil, sadece Reuters başlığıdır.
annotation_df["text_en"] = sample_df["headline_text"].astype(str).str.strip()
annotation_df["text_tr"] = ""

# Kontrol için ham metni ayrı saklamak istersen True yap
KEEP_RAW_TEXT = False

if KEEP_RAW_TEXT:
    annotation_df["raw_text"] = sample_df["raw_text"].astype(str).str.strip()

# FinBERT/pseudo label bilgileri
annotation_df["finbert_score"] = sample_df["finbert_score"].values
annotation_df["finbert_label"] = sample_df["finbert_label"].values
annotation_df["finbert_confidence"] = sample_df["finbert_confidence"].values

# Ek bilgiler
for col in [
    "primary_topic",
    "topics",
    "topic_confidence",
    "financial_score",
    "hour",
]:
    if col in sample_df.columns:
        annotation_df[col] = sample_df[col].values

annotation_df["n_words"] = annotation_df["text_en"].str.split().str.len()
annotation_df["text_len"] = annotation_df["text_en"].str.len()

# İnsan/ChatGPT annotation kolonları
annotation_df["chatgpt_label"] = ""
annotation_df["chatgpt_confidence"] = ""
annotation_df["chatgpt_reason_tr"] = ""
annotation_df["final_label"] = ""
annotation_df["annotation_note"] = ""
annotation_df["finbert_correctness"] = ""
annotation_df["finbert_correctness_note"] = ""

print("\nannotation_df shape:", annotation_df.shape)

print("\nAnnotation score distribution:")
print(annotation_df["finbert_score"].value_counts().sort_index())

print("\nAnnotation label distribution:")
print(annotation_df["finbert_label"].value_counts())

display(annotation_df.head(PREVIEW_ROWS))

# ------------------------------------------------------------
# 8) Master dosyayı kaydet
# ------------------------------------------------------------
OUT_XLSX = OUT_DIR / "reuters_annotation_5000_master.xlsx"
OUT_CSV = OUT_DIR / "reuters_annotation_5000_master.csv"
OUT_PARQUET = OUT_DIR / "reuters_annotation_5000_master.parquet"

# İstersen Excel/parquet kapalı kalabilir.
# annotation_df.to_excel(OUT_XLSX, index=False)
annotation_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
# annotation_df.to_parquet(OUT_PARQUET, index=False)

print("\nMaster dosyalar kaydedildi:")
print("CSV:", OUT_CSV)
print("Excel kapalı:", OUT_XLSX)
print("Parquet kapalı:", OUT_PARQUET)

# ------------------------------------------------------------
# 9) 50'şer batch dosyaları oluştur
# ------------------------------------------------------------
n_batches = math.ceil(len(annotation_df) / BATCH_SIZE)

for batch_no in range(n_batches):
    start = batch_no * BATCH_SIZE
    end = start + BATCH_SIZE

    batch_df = annotation_df.iloc[start:end].copy()
    batch_id = batch_no + 1

    batch_csv = BATCH_DIR / f"reuters_annotation_batch_{batch_id:03d}.csv"
    batch_txt = BATCH_DIR / f"reuters_annotation_batch_{batch_id:03d}.txt"

    batch_df.to_csv(batch_csv, index=False, encoding="utf-8-sig")

    with open(batch_txt, "w", encoding="utf-8") as f:
        f.write("=" * 120 + "\n")
        f.write(f"REUTERS ANNOTATION BATCH {batch_id:03d}\n")
        f.write(f"Rows: {start} - {start + len(batch_df) - 1}\n")
        f.write("=" * 120 + "\n\n")

        for _, row in batch_df.iterrows():
            f.write("-" * 120 + "\n")
            f.write(f"annotation_id: {row.get('annotation_id', '')}\n")
            f.write(f"date: {row.get('date', '')}\n")
            f.write(f"finbert_score: {row.get('finbert_score', '')}\n")
            f.write(f"finbert_label: {row.get('finbert_label', '')}\n")
            f.write(f"finbert_confidence: {row.get('finbert_confidence', '')}\n")
            f.write("TEXT:\n")
            f.write(str(row.get("text_en", "")) + "\n\n")

print("\nBatch dosyaları oluşturuldu.")
print("Batch sayısı:", n_batches)
print("Batch klasörü:", BATCH_DIR)

# ------------------------------------------------------------
# 10) Son kontrol: full article gelmiş mi?
# ------------------------------------------------------------
print("\nSon kontrol - ilk 3 başlık:")
for i, row in annotation_df.head(PREVIEW_ROWS).iterrows():
    print("=" * 100)
    print(row["annotation_id"], "|", row["finbert_label"], "|", row["finbert_confidence"])
    print(row["text_en"])

OUT_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000
BATCH_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50
Raw reuters_df shape: (6433, 12)
Headline Reuters shape: (6083, 18)


,date,headline_text,sentiment,sentiment_score,sentiment_confidence,n_words,text_len
0,2006-10-20 00:00:00,exxon mobil offers plan to end alaska dispute,positive,1.0,0.555539,8,45
1,2006-10-20 00:00:00,"hey buddy, can you spare $600 for a google share",positive,1.0,0.919182,10,48
2,2006-10-21 00:00:00,aol ceo says sales may shrink for two years -p...,negative,-1.0,0.967526,10,50



Reuters headline score distribution:
finbert_score
-1    2288
 0    1507
 1    2288
Name: count, dtype: int64

Reuters headline label distribution:
finbert_label
positive    2288
negative    2288
neutral     1507
Name: count, dtype: int64

Available score counts:
finbert_score
-1    2288
 0    1507
 1    2288
Name: count, dtype: int64

Target counts:
{-1: 1746, 0: 1507, 1: 1747}

Selected sample shape: (5000, 21)

Selected score distribution:
finbert_score
-1    1746
 0    1507
 1    1747
Name: count, dtype: int64

Selected label distribution:
finbert_label
positive    1747
negative    1746
neutral     1507
Name: count, dtype: int64

annotation_df shape: (5000, 24)

Annotation score distribution:
finbert_score
-1    1746
 0    1507
 1    1747
Name: count, dtype: int64

Annotation label distribution:
finbert_label
positive    1747
negative    1746
neutral     1507
Name: count, dtype: int64


,annotation_id,source_dataset,source_row_index,date,date_only,text_en,text_tr,finbert_score,finbert_label,finbert_confidence,...,hour,n_words,text_len,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,final_label,annotation_note,finbert_correctness,finbert_correctness_note
0,REUTERS_ANN_00001,reuters_news,5392,2007-04-12,2007-04-12,s.africa watchdog to probe gold fields bid report,,-1,negative,0.683211,...,0,8,49,,,,,,,
1,REUTERS_ANN_00002,reuters_news,5955,2007-04-26,2007-04-26,new barbie girls sashay into view with mp-3,,0,neutral,0.847032,...,0,8,43,,,,,,,
2,REUTERS_ANN_00003,reuters_news,1709,2006-12-17,2006-12-17,"stocks await data, mergers and santa",,0,neutral,0.865116,...,0,6,36,,,,,,,



Master dosyalar kaydedildi:
CSV: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\reuters_annotation_5000_master.csv
Excel kapalı: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\reuters_annotation_5000_master.xlsx
Parquet kapalı: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\reuters_annotation_5000_master.parquet

Batch dosyaları oluşturuldu.
Batch sayısı: 100
Batch klasörü: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50

Son kontrol - ilk 3 başlık:
REUTERS_ANN_00001 | negative | 0.683210551738739
s.africa watchdog to probe gold fields bid report
REUTERS_ANN_00002 | neutral | 0.8470315337181091
new barbie girls sashay into view with mp-3
REUTERS_ANN_00003 | neutral | 0.8651162981987
stocks await data, mergers and santa
